# Lab 6: Distilling a writer

The task changes: for each customer message, write a ticket title of at most eight words.
You will distill the 7B teacher's title writing into the 0.5B student, two ways.

1. The teacher writes titles; rule-breaking ones are dropped.
2. The teacher's top-20 tokens are recorded at every position of every title.
3. Baseline: the untrained student, with full instructions.
4. A student trained on the teacher's titles (sequence-level).
5. A student trained on the teacher's token probabilities as well (token-level).
6. Everything is scored with rules, overlap, and the teacher as judge.

This lab does not need results from earlier labs. Allow about 50 minutes. Run the cells in order.

In [ ]:
from kd_course.env import check_environment

report = check_environment()
print(report.summary())
assert report.ok, "Fix the problems listed above before continuing."

N_TRAIN = 2000  # pool messages the teacher writes titles for
N_TEST = 300  # held-out messages for scoring

In [ ]:
from kd_course.data import load_banking77, make_splits

bundle = load_banking77()
splits = make_splits(bundle.train, bundle.test)
train_messages = splits.pool.texts[:N_TRAIN]
test_messages = splits.test.texts[:N_TEST]

## 1. The teacher writes titles

In [ ]:
from kd_course.models import STUDENT_ID, TEACHER_ID, load_causal_lm
from kd_course.teachers import LocalTeacher
from kd_course.titles import generate_titles, usable_for_training

teacher_model, tokenizer = load_causal_lm(TEACHER_ID)
teacher = LocalTeacher(teacher_model, tokenizer)

raw_titles = generate_titles(teacher, train_messages)
teacher_test_titles = generate_titles(teacher, test_messages)

keep = [
    i
    for i, (m, t) in enumerate(zip(train_messages, raw_titles, strict=True))
    if usable_for_training(m, t)
]
kept_messages = [train_messages[i] for i in keep]
kept_titles = [raw_titles[i] for i in keep]
print(f"Kept {len(keep)} of {len(raw_titles)} titles; dropped {len(raw_titles) - len(keep)}.")
for message, title in list(zip(kept_messages, kept_titles, strict=True))[:5]:
    print(f"- {message}\n  -> {title}")

## 2. Record the teacher's top-20 tokens along each title

The teacher reads its own prompt and its own title in one pass. Nothing is generated.

In [ ]:
from kd_course.titles import student_prompt, teacher_chat_prompts
from kd_course.token_kd import storage_gb, topk_targets

records = topk_targets(
    teacher_model,
    tokenizer,
    teacher_chat_prompts(tokenizer, kept_messages),
    kept_titles,
    k=20,
    answer_prefix="",
)
positions = sum(len(r.target_ids) for r in records)
print(f"{positions:,} title tokens recorded.")
print(f"Top-20 storage:          {storage_gb(positions, 20) * 1024:.1f} MB")
print(f"Full-vocabulary storage: {storage_gb(positions, 20, len(tokenizer)):.1f} GB")

In [ ]:
import gc

import torch


def free():
    """Release GPU memory. Delete your own references to a model before calling this."""
    gc.collect()
    torch.cuda.empty_cache()


del teacher, teacher_model
free()

## 3. Baseline: the untrained student

It gets the same full instructions as the teacher. Nobody has trained it to write titles.

In [ ]:
titles = {"teacher": teacher_test_titles}

student, student_tokenizer = load_causal_lm(STUDENT_ID)
assert student_tokenizer.get_vocab() == tokenizer.get_vocab(), (
    "Teacher and student tokenizers differ."
)

titles["untrained student"] = generate_titles(
    LocalTeacher(student, student_tokenizer), test_messages
)
del student
free()

## 4. Sequence-level student

Trained on the teacher's titles. Short prompt in, title out.

In [ ]:
from kd_course.generative import generate_replies, train_text_pairs
from kd_course.titles import clean_title

train_prompts = [student_prompt(m) for m in kept_messages]
test_prompts = [student_prompt(m) for m in test_messages]


def write_titles(model):
    replies = generate_replies(model, student_tokenizer, test_prompts, max_new_tokens=24)
    return [clean_title(reply) for reply in replies]


student, _ = load_causal_lm(STUDENT_ID, dtype="float32")
history = train_text_pairs(
    student, student_tokenizer, train_prompts, kept_titles, epochs=3, answer_prefix=""
)
print(history)
titles["sequence-level student"] = write_titles(student)
del student
free()

## 5. Token-level student

Same messages, same titles. The student also sees which other tokens the teacher considered
at each position.

In [ ]:
from kd_course.token_kd import train_token_kd

student, _ = load_causal_lm(STUDENT_ID, dtype="float32")
history = train_token_kd(
    student, student_tokenizer, train_prompts, records, temperature=2.0, alpha=0.5, epochs=3
)
print(history)
titles["token-level student"] = write_titles(student)
del student
free()

## 6. Rules and overlap

These need no model.

In [ ]:
from kd_course.titles import score_titles

scores = {name: score_titles(written, teacher_test_titles) for name, written in titles.items()}
for name, row in scores.items():
    print(
        f"{name:<24} within limit {row['within_limit']:.0%}   "
        f"overlap with teacher {row['overlap_with_teacher']:.0%}"
    )

## 7. The judge

The teacher is loaded again and asked, for every title: is this a good title for this message?
Remember it is judging its own work as well as the students'.

In [ ]:
from kd_course.results import default_results_dir, save_result
from kd_course.titles import acceptance, judge_titles

judge_model, _ = load_causal_lm(TEACHER_ID)
judge = LocalTeacher(judge_model, tokenizer)

verdicts = {name: judge_titles(judge, test_messages, written) for name, written in titles.items()}
del judge, judge_model
free()

folder = default_results_dir() / "generation"
print(f"{'':<24}{'within limit':>14}{'overlap':>10}{'judge accepts':>16}{'no verdict':>12}")
for name in titles:
    scores[name].update(acceptance(verdicts[name]))
    save_result(name.replace(" ", "_").replace("-", "_"), scores[name], titles[name], folder)
    row = scores[name]
    print(
        f"{name:<24}{row['within_limit']:>14.0%}{row['overlap_with_teacher']:>10.0%}"
        f"{row['accepted']:>16.0%}{row['unanswered']:>12.0%}"
    )

## 8. Are the differences real?

Paired comparisons on the judge's verdicts.

In [ ]:
from kd_course.metrics import accuracy, paired_bootstrap

accepted = {name: [int(v is True) for v in verdict] for name, verdict in verdicts.items()}
ones = [1] * len(test_messages)


def report(title, a, b):
    r = paired_bootstrap(accuracy, ones, accepted[a], accepted[b])
    print(
        f"{title}: {r.difference * 100:+.1f} points "
        f"(95% interval {r.low * 100:+.1f} to {r.high * 100:+.1f})"
    )


report("Sequence-level minus untrained  ", "untrained student", "sequence-level student")
report("Token-level minus sequence-level", "sequence-level student", "token-level student")

## 9. Read some

Ten test messages with every model's title. Read these before trusting the table.

In [ ]:
for i in range(10):
    print(test_messages[i])
    for name, written in titles.items():
        mark = {True: "yes", False: "no ", None: " ? "}[verdicts[name][i]]
        print(f"   [{mark}] {name:<24} {written[i][:90]}")
    print()

## 10. What did you see?

Double-click this cell and write your answers.

**a. How often did each model stay within eight words? Did the trained students beat the teacher on this? Why is that possible?**

**b. Did training beat the untrained student on the judge's verdicts? Quote the interval.**

**c. Did token-level distillation beat sequence-level? Is the difference real? What does that say about this task?**

**d. What share of its own titles did the teacher, as judge, accept? What does that tell you about how to read the other rows?**

**e. In the ten examples you read, did you disagree with the judge anywhere? What would you change in the judge's question?**

**f. The student's prompt is a few tokens and the teacher's is about seventy. For a million titles a day, what does that difference mean in cost?**